In [1]:
import abc
from dataclasses import dataclass, replace
from decimal import Decimal, ROUND_HALF_UP, InvalidOperation
from datetime import date
from enum import Enum
from hypothesis import given, settings, strategies as st


In [2]:

# Custom errors (assignment requires a hierarchy)
class TaxError(Exception): pass
class InvalidIncomeError(TaxError): pass
class NoScheduleForDateError(TaxError): pass



In [3]:

class Money:
    """An amount of Ugandan shillings (UGX). Immutable: operations return NEW Money."""
    
    def __str__(self):
        return f"UGX {self.amount:,.2f}"

    def __init__(self, amount):
        # Reject float: 0.1 + 0.2 != 0.3 in binary floating point
        if isinstance(amount, float):
            raise TypeError("Money cannot be built from float. Use Decimal or str.")
        # Accept Decimal, int, or str ("1000.50"); anything else is an error
        if not isinstance(amount, (Decimal, int, str)):
            raise TypeError(f"Unsupported type for Money: {type(amount).__name__}")
        # Round ONCE, explicitly, to 2 decimal places
        self.amount = Decimal(amount).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented      # lets Python raise a clean TypeError
        return Money(self.amount + other.amount)

    def __sub__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return Money(self.amount - other.amount)

    def __mul__(self, factor):
        # Money * number (e.g. a tax rate). Money * Money makes no sense.
        if isinstance(factor, float) or not isinstance(factor, (Decimal, int)):
            return NotImplemented
        return Money(self.amount * factor)

    __rmul__ = __mul__                 # so Decimal("0.2") * money also works

    def __lt__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.amount < other.amount

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.amount == other.amount

    def __hash__(self):
        return hash(self.amount)

    def __repr__(self):
        # Developer-friendly: shows exactly what it holds
        return f"Money('{self.amount}' UGX)"

In [4]:
@dataclass(frozen=True)  # frozen = can't be changed after creation
class TaxContext:
    """Everything a tax needs as input."""
    gross_pay: Decimal
    allowances: Decimal = Decimal("0")  
    resident: bool = True
    on_date: date = date.today()

    @property
    def total_pay(self):
        return self.gross_pay+self.allowances

@dataclass(frozen=True)
class TaxResult:
    """What a tax calculation returns."""
    tax_name: str
    amount: Money

class Tax(abc.ABC):
    @abc.abstractmethod
    def calculate(self, ctx: TaxContext) -> TaxResult:
        """Every tax must implement this."""

In [5]:
D = Decimal

class PAYE(Tax):
    def calculate(self, ctx: TaxContext) -> TaxResult:
        income = ctx.total_pay

        if income < 0:
            raise InvalidIncomeError(f"Invalid income: {income}")

        if income <= 335000:
            tax = D("0")
        elif income <= 410000:
            tax = D("0.20") * (income - 335000)
        elif income <= 485000:
            tax = 15000 + D("0.25") * (income - 410000)
        else:
            # everything above 485,000 uses 30%
            tax = 33750 + D("0.30") * (income - 485000)

        # surcharge is added ON TOP, only above 10,000,000
        if income > 10_000_000:
            tax += D("0.10") * (income - 10_000_000)

        return TaxResult(tax_name="PAYE", amount=Money(tax))

In [6]:
a = Money("1000.10")
b = Money("2000.20")
print(a + b)                  # Money('3000.30' UGX)
print(a * Decimal("0.2"))     # Money('200.02' UGX)
print(a < b)                  # True
print(Decimal("0.1") + Decimal("0.2") == Decimal("0.3"))  # True
print(0.1 + 0.2 == 0.3)       # False  <- your answer to discussion question 1

UGX 3,000.30
UGX 200.02
True
True
False


In [7]:
class NSSF(Tax):
    """National Social Security Fund: employee 5%, employer 10% of gross."""
    EMPLOYEE_RATE = D("0.05")
    EMPLOYER_RATE = D("0.10")

    def calculate(self, ctx: TaxContext) -> TaxResult:
        if ctx.total_pay < 0:
            raise InvalidIncomeError(f"Invalid income: {ctx.total_pay}")
        return TaxResult("NSSF (employee)", Money(ctx.total_pay) * self.EMPLOYEE_RATE)

    def employer_contribution(self, ctx: TaxContext) -> Money:
        # Paid by the employer on top; it is NOT deducted from the worker
        return Money(ctx.total_pay) * self.EMPLOYER_RATE


class LocalServiceTax(Tax):
    """Configurable flat monthly amount. Default 0 because it's optional."""
    def __init__(self, monthly_amount="0"):
        self.monthly_amount = Money(monthly_amount)

    def calculate(self, ctx: TaxContext) -> TaxResult:
        return TaxResult("Local Service Tax", self.monthly_amount)

In [8]:
@dataclass(frozen=True)
class Payslip:
    gross: Money
    nssf: Money
    lst: Money
    paye: Money
    net: Money

class Payroll:
    """Runs the deductions in a fixed, documented order."""
    def __init__(self, lst_monthly="0"):
        self.nssf = NSSF()
        self.lst = LocalServiceTax(lst_monthly)
        self.paye = PAYE()

    def run(self, ctx: TaxContext) -> Payslip:
        gross = Money(ctx.total_pay)

        # Step 1: NSSF, calculated on gross
        nssf = self.nssf.calculate(ctx).amount

        # Step 2: Local Service Tax, deducted BEFORE PAYE (per the spec)
        lst = self.lst.calculate(ctx).amount

        # Step 3: PAYE on what is left after LST
        chargeable = ctx.total_pay - lst.amount
        paye_ctx = replace(ctx, gross_pay=chargeable, allowances=D("0"))
        paye = self.paye.calculate(paye_ctx).amount

        # Step 4: net pay
        net = gross - nssf - lst - paye
        return Payslip(gross, nssf, lst, paye, net)

In [9]:

class VATCategory(Enum):
    STANDARD = "standard"
    ZERO = "zero"
    EXEMPT = "exempt"

# One place for the rates; change it here if URA changes it
VAT_RATES = {
    VATCategory.STANDARD: D("0.18"),
    VATCategory.ZERO:     D("0"),
    VATCategory.EXEMPT:   D("0"),
}

@dataclass(frozen=True)
class Invoice:
    description: str
    net_amount: Money          # amount BEFORE VAT
    category: VATCategory
    kind: str                  # "sale" or "purchase"

    def __post_init__(self):
        # Runs right after creation: validate, never fail silently
        if self.kind not in ("sale", "purchase"):
            raise InvalidIncomeError(f"kind must be 'sale' or 'purchase', got {self.kind!r}")
        if not isinstance(self.net_amount, Money):
            raise InvalidIncomeError("net_amount must be a Money object")
        if self.net_amount < Money("0"):
            raise InvalidIncomeError(f"Negative invoice amount: {self.net_amount}")

@dataclass(frozen=True)
class VATContext:
    invoices: tuple            # the list of invoices to process
    on_date: date = date.today()

@dataclass(frozen=True)
class VATSummary:
    output_vat: Money
    input_vat: Money
    net_payable: Money

In [10]:
class VAT(Tax):
    def summary(self, ctx: VATContext) -> VATSummary:
        if not ctx.invoices:
            raise InvalidIncomeError("No invoices supplied.")

        output_vat = Money("0")
        input_vat = Money("0")

        for inv in ctx.invoices:
            vat = inv.net_amount * VAT_RATES[inv.category]
            if inv.kind == "sale":
                output_vat = output_vat + vat      # VAT we charged
            else:
                input_vat = input_vat + vat        # VAT we paid

        return VATSummary(output_vat, input_vat, output_vat - input_vat)

    def calculate(self, ctx: VATContext) -> TaxResult:
        # Satisfies the abstract method: returns ONE TaxResult
        return TaxResult("VAT payable", self.summary(ctx).net_payable)

In [11]:
@dataclass(frozen=True)
class Deduction:
    description: str
    amount: Money

    def __post_init__(self):
        if not isinstance(self.amount, Money):
            raise InvalidIncomeError("Deduction amount must be a Money object")
        if self.amount < Money("0"):
            raise InvalidIncomeError(f"Negative deduction: {self.amount}")

@dataclass(frozen=True)
class CITContext:
    revenue: Money
    deductions: tuple = ()          # tuple of Deduction objects
    resident: bool = True
    on_date: date = date.today()

@dataclass(frozen=True)
class CITSummary:
    revenue: Money
    total_deductions: Money
    chargeable_income: Money
    tax: Money

In [12]:
class CorporateIncomeTax(Tax):
    RATE = D("0.30")   # resident companies (check the URA source for the version)

    def summary(self, ctx: CITContext) -> CITSummary:
        # 1. Validate: never fail silently
        if not isinstance(ctx.revenue, Money):
            raise InvalidIncomeError("Revenue must be a Money object")
        if ctx.revenue < Money("0"):
            raise InvalidIncomeError(f"Negative revenue: {ctx.revenue}")

        # 2. The baseline only gives a rate for RESIDENT companies.
        #    Refuse to guess a rate for anyone else.
        if not ctx.resident:
            raise TaxError("No schedule defined for non-resident companies.")

        # 3. Add up all allowable deductions
        total_ded = Money("0")
        for d in ctx.deductions:
            total_ded = total_ded + d.amount

        # 4. Chargeable income = revenue - deductions
        chargeable = ctx.revenue - total_ded

        # 5. A loss is taxed at zero, never negative
        if chargeable < Money("0"):
            tax = Money("0")
            chargeable = Money("0")
        else:
            tax = chargeable * self.RATE

        return CITSummary(ctx.revenue, total_ded, chargeable, tax)

    def calculate(self, ctx: CITContext) -> TaxResult:
        return TaxResult("Corporate income tax", self.summary(ctx).tax)

In [13]:
ctx = CITContext(
    revenue=Money("50000000"),
    deductions=(
        Deduction("Salaries", Money("12000000")),
        Deduction("Rent and utilities", Money("8000000")),
    ),
)
s = CorporateIncomeTax().summary(ctx)
print("Revenue          :", s.revenue)            # 50,000,000.00
print("Deductions       :", s.total_deductions)   # 20,000,000.00
print("Chargeable income:", s.chargeable_income)  # 30,000,000.00
print("CIT (30%)        :", s.tax)                # 9,000,000.00

# Loss case: deductions bigger than revenue
loss = CITContext(Money("1000000"), (Deduction("Big expense", Money("3000000")),))
print(CorporateIncomeTax().summary(loss).tax)     # 0.00

Revenue          : UGX 50,000,000.00
Deductions       : UGX 20,000,000.00
Chargeable income: UGX 30,000,000.00
CIT (30%)        : UGX 9,000,000.00
UGX 0.00


In [14]:
def paye_amount(pay, resident=True):
    """Helper: PAYE tax for a gross pay, as a plain Decimal."""
    ctx = TaxContext(gross_pay=D(str(pay)), resident=resident)
    return PAYE().calculate(ctx).amount.amount

# (gross pay, expected PAYE). Each one is worked out by hand.
boundary_cases = [
    ("335000",    "0.00"),         # top of the nil band
    ("335001",    "0.20"),         # 20% of 1
    ("410000",    "15000.00"),     # 20% of 75,000
    ("410001",    "15000.25"),     # 25% of 1 on top
    ("485000",    "33750.00"),     # 15,000 + 25% of 75,000
    ("485001",    "33750.30"),     # 30% of 1 on top
    ("10000000",  "2888250.00"),   # 33,750 + 30% of 9,515,000
    ("10000001",  "2888250.40"),   # 30% + 10% surcharge on the extra 1
]

for pay, expected in boundary_cases:
    actual = paye_amount(pay)
    assert actual == D(expected), f"FAIL at {pay}: got {actual}, expected {expected}"
    print(f"PASS  {pay:>10}  ->  {actual:>14,}")

# Errors must be raised, never silent
for bad in ["-1", "-500000"]:
    try:
        paye_amount(bad)
        raise AssertionError(f"No error raised for {bad}")
    except InvalidIncomeError:
        print(f"PASS  {bad:>10}  ->  InvalidIncomeError raised")

PASS      335000  ->            0.00
PASS      335001  ->            0.20
PASS      410000  ->       15,000.00
PASS      410001  ->       15,000.25
PASS      485000  ->       33,750.00
PASS      485001  ->       33,750.30
PASS    10000000  ->    2,888,250.00
PASS    10000001  ->    2,888,250.40
PASS          -1  ->  InvalidIncomeError raised
PASS     -500000  ->  InvalidIncomeError raised


In [15]:

# Random incomes from 0 to 50 billion, 2 decimal places
incomes = st.decimals(min_value=0, max_value=50_000_000_000, places=2,
                      allow_nan=False, allow_infinity=False)

@settings(deadline=None)          # no time limit per example
@given(incomes)
def test_never_negative(x):
    assert paye_amount(x) >= 0

@settings(deadline=None)
@given(incomes)
def test_never_exceeds_income(x):
    assert paye_amount(x) <= x

@settings(deadline=None)
@given(incomes, incomes)
def test_monotonic(a, b):
    # More income must never mean LESS tax
    low, high = min(a, b), max(a, b)
    assert paye_amount(low) <= paye_amount(high)

test_never_negative()
test_never_exceeds_income()
test_monotonic()
print("All property tests passed.")

All property tests passed.


In [16]:
# Payroll: gross 1,000,000 -> NSSF 50,000, PAYE 188,250, net 761,750
slip = Payroll().run(TaxContext(gross_pay=D("1000000")))
assert slip.nssf == Money("50000")
assert slip.paye == Money("188250")
assert slip.net == Money("761750")
print("PASS  payroll")

# VAT: output 180,000, input 72,000, net 108,000
S, Z, E = VATCategory.STANDARD, VATCategory.ZERO, VATCategory.EXEMPT
inv = (Invoice("a", Money("1000000"), S, "sale"),
       Invoice("b", Money("500000"),  Z, "sale"),
       Invoice("c", Money("200000"),  E, "sale"),
       Invoice("d", Money("400000"),  S, "purchase"))
r = VAT().summary(VATContext(inv))
assert (r.output_vat, r.input_vat, r.net_payable) == (Money("180000"), Money("72000"), Money("108000"))
print("PASS  VAT")

# CIT: 50M revenue, 20M deductions -> 9M tax; a loss -> 0
c = CITContext(Money("50000000"), (Deduction("x", Money("20000000")),))
assert CorporateIncomeTax().calculate(c).amount == Money("9000000")
loss = CITContext(Money("1000000"), (Deduction("x", Money("3000000")),))
assert CorporateIncomeTax().calculate(loss).amount == Money("0")
print("PASS  CIT")

# Money rejects float
try:
    Money(0.1)
    raise AssertionError("float was accepted!")
except TypeError:
    print("PASS  Money rejects float")

PASS  payroll
PASS  VAT
PASS  CIT
PASS  Money rejects float


In [17]:
def payroll_calculator():
    while True:
        raw = input("Monthly gross pay in UGX (or 'q' to quit): ").strip()
        if raw.lower() == "q":
            break
        try:
            gross = Decimal(raw.replace(",", ""))
            allow = Decimal(input("Allowances (0 if none): ").replace(",", "") or "0")
            lst = input("Local Service Tax this month (0 if none): ").replace(",", "") or "0"
        except InvalidOperation:
            print("That's not a number. Try again.\n")
            continue
        resident = input("Resident? (y/n): ").strip().lower() == "y"

        try:
            ctx = TaxContext(gross_pay=gross, allowances=allow, resident=resident)
            slip = Payroll(lst_monthly=lst).run(ctx)
            print(f"\nGross : {slip.gross}")
            print(f"NSSF  : {slip.nssf}")
            print(f"LST   : {slip.lst}")
            print(f"PAYE  : {slip.paye}")
            print(f"NET   : {slip.net}\n")
        except TaxError as e:
            print(f"Error: {e}\n")

payroll_calculator()

Monthly gross pay in UGX (or 'q' to quit):  q


In [18]:
def vat_calculator():
    print("VAT CALCULATOR")
    print("Add your sales and purchases one at a time.\n")

    # Running totals; each entry adds to one of these
    totals = {"1": Money("0"), "2": Money("0"), "3": Money("0"), "4": Money("0")}

    while True:
        print("What do you want to add?")
        print("  1 = Standard-rated sale (18% VAT)")
        print("  2 = Zero-rated sale (e.g. exports)")
        print("  3 = Exempt sale")
        print("  4 = Purchase with VAT (standard-rated)")
        print("  5 = Finish and show my VAT")
        choice = input("Your choice (1-5): ").strip()

        if choice == "5":
            break
        if choice not in totals:
            print("Please type a number from 1 to 5.\n")
            continue

        text = input("Enter the amount BEFORE VAT, in UGX: ").replace(",", "").strip()
        try:
            totals[choice] = totals[choice] + Money(text)
            print(f"Added. Total for this type so far: {totals[choice]}\n")
        except (TaxError, ArithmeticError, TypeError):
            print("That is not a valid amount. Please try again.\n")

    # Turn the four totals into invoices so the VAT class works unchanged
    invoices = (
        Invoice("Standard sales", totals["1"], S, "sale"),
        Invoice("Zero sales",     totals["2"], Z, "sale"),
        Invoice("Exempt sales",   totals["3"], E, "sale"),
        Invoice("Purchases",      totals["4"], S, "purchase"),
    )
    try:
        r = VAT().summary(VATContext(invoices))
        print("\n----- YOUR VAT -----")
        print(f"Output VAT (charged on sales)   : {r.output_vat}")
        print(f"Input VAT (paid on purchases)   : {r.input_vat}")
        print(f"NET VAT PAYABLE                 : {r.net_payable}")
        if r.net_payable < Money("0"):
            print("(Negative means a VAT refund or credit.)")
    except TaxError as e:
        print(f"Error: {e}")

vat_calculator()

VAT CALCULATOR
Add your sales and purchases one at a time.

What do you want to add?
  1 = Standard-rated sale (18% VAT)
  2 = Zero-rated sale (e.g. exports)
  3 = Exempt sale
  4 = Purchase with VAT (standard-rated)
  5 = Finish and show my VAT


Your choice (1-5):  5



----- YOUR VAT -----
Output VAT (charged on sales)   : UGX 0.00
Input VAT (paid on purchases)   : UGX 0.00
NET VAT PAYABLE                 : UGX 0.00


In [19]:
def cit_calculator():
    print("Corporate Income Tax (resident company, 30%)\n")
    try:
        revenue = Money(input("Total revenue (UGX): ").replace(",", "").strip())
        # Press Enter to skip if there are no deductions
        ded_text = input("Total allowable deductions (UGX, press Enter for 0): ")
        ded_text = ded_text.replace(",", "").strip() or "0"
        total_ded = Money(ded_text)

        # Wrap the total in ONE Deduction so the class works unchanged
        ctx = CITContext(revenue, (Deduction("Total deductions", total_ded),))
        s = CorporateIncomeTax().summary(ctx)

        print(f"\nRevenue          : {s.revenue}")
        print(f"Deductions       : {s.total_deductions}")
        print(f"Chargeable income: {s.chargeable_income}")
        print(f"CIT (30%)        : {s.tax}")
    except (TaxError, ArithmeticError, TypeError) as e:
        print(f"Error: {e}")

cit_calculator()

Corporate Income Tax (resident company, 30%)



Total revenue (UGX):  0
Total allowable deductions (UGX, press Enter for 0):  



Revenue          : UGX 0.00
Deductions       : UGX 0.00
Chargeable income: UGX 0.00
CIT (30%)        : UGX 0.00


In [20]:
@dataclass(frozen=True)
class ScenarioResult:
    name: str
    gross: Money
    total_deductions: Money
    net: Money

class Scenario(abc.ABC):
    """Any way of receiving money. Must be able to evaluate itself."""
    @abc.abstractmethod
    def evaluate(self) -> ScenarioResult:
        ...

class SalaryScenario(Scenario):
    """Money received as monthly salary: NSSF + Local Service Tax + PAYE."""
    def __init__(self, gross_pay, resident=True, lst_monthly="0"):
        self.gross_pay = D(str(gross_pay))
        self.resident = resident
        self.lst_monthly = lst_monthly

    def evaluate(self) -> ScenarioResult:
        ctx = TaxContext(gross_pay=self.gross_pay, resident=self.resident)
        slip = Payroll(self.lst_monthly).run(ctx)      # reuses Step 4
        total = slip.nssf + slip.lst + slip.paye
        return ScenarioResult("Salary", slip.gross, total, slip.net)

class DividendScenario(Scenario):
    """Money received as a dividend: a flat withholding tax, no PAYE."""
    # ASSUMPTION: verify the current rate on the URA site and cite it.
    DEFAULT_WHT = D("0.15")

    def __init__(self, amount, wht_rate=None):
        self.amount = D(str(amount))
        self.wht_rate = self.DEFAULT_WHT if wht_rate is None else D(str(wht_rate))

    def evaluate(self) -> ScenarioResult:
        if self.amount < 0:
            raise InvalidIncomeError(f"Invalid dividend: {self.amount}")
        if not (D("0") <= self.wht_rate <= D("1")):
            raise InvalidIncomeError(f"Rate must be between 0 and 1: {self.wht_rate}")
        gross = Money(self.amount)
        tax = gross * self.wht_rate
        return ScenarioResult("Dividend", gross, tax, gross - tax)

In [21]:
@dataclass(frozen=True)
class Comparison:
    first: ScenarioResult
    second: ScenarioResult
    net_difference: Money      # first.net - second.net (can be negative)
    better: str                # name of the scenario that leaves more net money

class TaxPlanner:
    def compare(self, first: Scenario, second: Scenario) -> Comparison:
        a = first.evaluate()
        b = second.evaluate()
        diff = a.net - b.net

        if b.net < a.net:
            better = a.name
        elif a.net < b.net:
            better = b.name
        else:
            better = "Equal"
        return Comparison(a, b, diff, better)

In [22]:
c = TaxPlanner().compare(SalaryScenario("1000000"), DividendScenario("1000000"))

# Salary:   NSSF 50,000 + PAYE 188,250 = 238,250 -> net 761,750
# Dividend: 15% of 1,000,000 = 150,000           -> net 850,000
assert c.first.net  == Money("761750")
assert c.second.net == Money("850000")
assert c.net_difference == Money("-88250")
assert c.better == "Dividend"
print("PASS  TaxPlanner")

PASS  TaxPlanner


In [23]:
def planner_calculator():
    print("SALARY vs DIVIDEND")
    print("Compare receiving the SAME amount each way.\n")
    try:
        text = input("Amount to compare (UGX): ").replace(",", "").strip()
        amount = D(text)
        resident = input("Resident? (y/n): ").strip().lower() == "y"

        c = TaxPlanner().compare(SalaryScenario(amount, resident),
                                 DividendScenario(amount))

        print(f"\n{'':10}{'Gross':>18}{'Deductions':>18}{'Net':>18}")
        for r in (c.first, c.second):
            print(f"{r.name:10}{str(r.gross):>18}{str(r.total_deductions):>18}{str(r.net):>18}")

        gap = c.net_difference if c.net_difference > Money("0") else Money("0") - c.net_difference
        print(f"\n{c.better} leaves you better off by {gap}.")
    except (TaxError, ArithmeticError) as e:
        print(f"Error: {e}")

planner_calculator()

SALARY vs DIVIDEND
Compare receiving the SAME amount each way.



Amount to compare (UGX):  500000
Resident? (y/n):  y



                       Gross        Deductions               Net
Salary        UGX 500,000.00     UGX 63,250.00    UGX 436,750.00
Dividend      UGX 500,000.00     UGX 75,000.00    UGX 425,000.00

Salary leaves you better off by UGX 11,750.00.


Why is Decimal used instead of float?
Floats store most decimal numbers as close approximations, not exact values. 0.1 + 0.2 == 0.3 is False with floats, and True with Decimal. for decimals its  the exact while for floats,they are kind of not the same its 0.30000000000000004. furthermore, rounding can go the wrong way: round(2.675, 2) gives 2.67 with float, while Decimal with ROUND_HALF_UP gives the correct 2.68. When small errors add up across many payslips, so totals drift from what URA expects.
Decimal stores exact decimal digits, so money calculations stay accurate.

Where would inheritance be justified, and where did we avoid it?

Where we used it:
Tax is an abstract base class. PAYE, NSSF, LocalServiceTax, VAT and CorporateIncomeTax all inherit from it, because each one is a kind of tax with a calculate method.
Scenario is the base class for SalaryScenario and DividendScenario, for the same reason.

Where we avoided:
Money does not inherit from Decimal. It holds a Decimal inside. This lets us block float input and nonsense operations like Money * Money.
There are no ResidentPAYE or NonResidentPAYE subclasses. Residency is just a value (resident=True/False) in TaxContext, so it doesn’t need a new class.
NSSF does not inherit from PAYE, even though both take a percentage. They share a purpose, not logic.
Payroll does not inherit from the taxes. It uses them, since a payroll has an NSSF and a PAYE; it is not one of them.